# Compare plates
Run after `01_process_plates` has been run for both plates.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from plate_utils import CONDITION_ORDER
%matplotlib inline

In [ ]:
OUT_DIR = r"C:\Users\sam\Desktop\plate_assay\results"
PLATES = ["plate1", "plate2"]

In [ ]:
boots = []
for p in PLATES:
    b = pd.read_csv(os.path.join(OUT_DIR, f"bootstrap_{p}.csv"))
    b["plate"] = p
    boots.append(b)
combined = pd.concat(boots)
# drop conditions with too few cells to trust
combined_clean = combined[combined["n_cells"] >= 50]

In [ ]:
summary = combined.pivot(index="condition", columns="plate", values="mean_fc").reindex(CONDITION_ORDER)
summary["mean_across_plates"] = summary[PLATES].mean(axis=1)
summary["diff"] = summary["plate2"] - summary["plate1"]
summary.to_csv(os.path.join(OUT_DIR, "plate_comparison.csv"))
summary

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(summary["plate1"], summary["plate2"])
for cond, row in summary.iterrows():
    ax.annotate(cond, (row["plate1"], row["plate2"]), fontsize=8)
lim = [0, max(summary["plate1"].max(), summary["plate2"].max()) * 1.1]
ax.plot(lim, lim, color="grey", lw=0.8, ls="--")
ax.set_xlabel("plate1 fold change")
ax.set_ylabel("plate2 fold change")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "comparison.png"), dpi=120)
plt.show()